# Twitter US Airline Sentiment – Simple RNN
Classify February 2015 airline tweets into negative, neutral and positive sentiment. Download Tweets.csv from [Kaggle](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment) and place it beside this notebook. Run all cells in order. TensorFlow, pandas, scikit-learn and matplotlib are required.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

tf.keras.utils.set_random_seed(42)

## 2. Load dataset and retain text and label

In [ ]:
df = pd.read_csv("Tweets.csv")
df = df[["text", "airline_sentiment"]].dropna().reset_index(drop=True)
df.head()

In [ ]:
print(df["airline_sentiment"].value_counts())
df["airline_sentiment"].value_counts().plot(kind="bar", title="Sentiment distribution")
plt.ylabel("Number of tweets")
plt.tight_layout()
plt.show()

## 3. Encode labels and split data
Use integer labels and sparse categorical cross-entropy. Hold out 20% for testing and 20% of the remaining data for validation.

In [ ]:
encoder = LabelEncoder()
X = df["text"]
y = encoder.fit_transform(df["airline_sentiment"])
n_classes = len(encoder.classes_)
print(dict(enumerate(encoder.classes_)))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

## 4. Tokenize and pad
Fit the tokenizer only on training tweets to avoid data leakage. Use the default pre-padding and pre-truncation to produce exactly 50 tokens per tweet.

In [ ]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=50)
X_val = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=50)
X_test = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=50)
print(X_train.shape, X_val.shape, X_test.shape)

## 5. Build the exact Simple RNN architecture

In [ ]:
model = Sequential([
    Embedding(5000, 32),
    SimpleRNN(32),
    Dense(n_classes, activation="softmax")
])
model.build((None, 50))
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Train for five epochs

In [ ]:
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_val, y_val))

## 7. Evaluate test accuracy

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy:.2%})")
print(f"Test loss: {test_loss:.4f}")

## 8. Training vs. validation curves

In [ ]:
epochs = range(1, 6)
plt.plot(epochs, history.history["accuracy"], marker="o", label="Training accuracy")
plt.plot(epochs, history.history["val_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.xticks(list(epochs))
plt.legend()
plt.title("Training vs. validation accuracy")
plt.show()

In [ ]:
plt.plot(epochs, history.history["loss"], marker="o", label="Training loss")
plt.plot(epochs, history.history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(list(epochs))
plt.legend()
plt.title("Training vs. validation loss")
plt.show()

## 9. Optional: classification report
The classes may be imbalanced, so inspect per-class metrics alongside accuracy.

In [ ]:
from sklearn.metrics import classification_report
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test, y_pred, labels=np.arange(n_classes), target_names=encoder.classes_, zero_division=0))

## 10. Vanishing gradient problem
During backpropagation through many time steps, repeatedly multiplying small gradients can make them approach zero. As a result, a plain RNN struggles to learn long-term dependencies between words that are far apart in a sequence.